# Opus Lower — ConvNeXtV2-Base (384px) Training
Lower garment sub-classification: Jeans, Trousers, Jogging_Bottoms, Skirts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, json, random, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, Sampler
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
ROOT = "/content/drive/Shareddrives/Garment Type/Complete_dataset"
OUTDIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ============================================================
# CHANGES FROM V2-TINY LOWER SCRIPT
# ============================================================
# 1. Backbone: convnextv2_tiny → convnextv2_base (89M vs 28M)
# 2. Resolution: 448 → 384 (native for Base-384 variant)
# 3. Batch size: 48 → 16 (larger model, AMP keeps VRAM manageable)
# 4. Backbone LR: 1e-5 → 5e-6 (gentler — bigger model, richer features)
# 5. Head LR: 1.5e-4 → 1e-4
# 6. Val batch: 64 → 24 (VRAM constraint)
# 7. Head epochs: 8 → 10 (more warmup for bigger head)
# ============================================================

# IMAGE SIZE — native resolution for Base-384 variant
IMG_SIZE = 384

# BACKBONE — ConvNeXt-V2 Base with FCMAE + IN-22k pretraining
BACKBONE = "convnextv2_base.fcmae_ft_in22k_in1k_384"

# TRAINING PARAMS
BATCH = 16              # reduced for 89M model at 384px
EPOCHS = 100
HEAD_EPOCHS = 10        # more warmup for bigger classifier head

LR_HEAD = 5e-4          # Stage 1 head-only LR
BACKBONE_LR = 5e-6      # Stage 2 backbone LR — gentler for Base
HEAD_LR = 1e-4           # Stage 2 head LR
WEIGHT_DECAY = 1e-2

PATIENCE = 15
WARMUP_EPOCHS = 5
NUM_WORKERS = 2

MIN_SAMPLES_FOR_BALANCED = 12

# LOSS SETTINGS
LABEL_SMOOTH = 0.05
FOCAL_GAMMA = 2.0

# MIXUP
MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.3         # conservative — let Base learn clean features first

# VAL SPLIT
VAL_RATIO = 0.16

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

LOWER_CLASSES = ["jeans", "jogging_bottoms", "skirts", "trousers"]

MINORITY_CLASSES = {"skirts"}
MEDIUM_CLASSES = {"jeans"}   # jeans is mid-sized (702 vs 1847 trousers)


# ============================================================
# FOCAL LOSS + LABEL SMOOTHING
# ============================================================
def focal_loss_with_ls(logits, targets, gamma=2.0, smooth=0.05, weight=None):
    num_classes = logits.shape[1]
    one_hot = F.one_hot(targets, num_classes).float().to(logits.device)
    one_hot = one_hot * (1 - smooth) + smooth / num_classes

    log_probs = F.log_softmax(logits, dim=1)
    probs = torch.exp(log_probs)

    focal_factor = (1 - probs) ** gamma
    loss = -one_hot * focal_factor * log_probs
    loss = loss.sum(dim=1)

    if weight is not None:
        loss = loss * weight[targets]

    return loss.mean()


def focal_loss_mixup(logits, targets_a, targets_b, lam, gamma=2.0, smooth=0.05, weight=None):
    loss_a = focal_loss_with_ls(logits, targets_a, gamma, smooth, weight)
    loss_b = focal_loss_with_ls(logits, targets_b, gamma, smooth, weight)
    return lam * loss_a + (1 - lam) * loss_b


# ============================================================
# MIXUP
# ============================================================
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    lam = max(lam, 1 - lam)

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


# ============================================================
# UTILITIES
# ============================================================
def collect_samples(root, allowed_classes):
    root = Path(root)
    samples = []

    for dataset_dir in root.iterdir():
        if not dataset_dir.is_dir():
            continue
        for cls_dir in dataset_dir.iterdir():
            if not cls_dir.is_dir():
                continue
            cls_name = cls_dir.name.lower()
            if cls_name not in allowed_classes:
                continue
            for img_path in cls_dir.iterdir():
                if img_path.is_file():
                    samples.append((str(img_path), cls_name))

    return samples


# ============================================================
# AUGMENTATIONS
# ============================================================
def build_aug(img_size, is_train=True):
    h = w = img_size

    if not is_train:
        return A.Compose([
            A.Resize(h, w),
            A.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    return A.Compose([
        A.RandomResizedCrop(
            size=(h, w),
            scale=(0.75, 1.0),
            ratio=(0.75, 1.3),
            p=1.0
        ),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.04,
            scale_limit=0.08,
            rotate_limit=8,
            p=0.6
        ),
        A.ColorJitter(0.2, 0.2, 0.2, 0.05, p=0.6),
        A.GaussianBlur(3, p=0.2),
        A.CoarseDropout(
            max_holes=4,
            max_height=int(h * 0.10),
            max_width=int(w * 0.10),
            p=0.3
        ),
        A.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


def build_strong_aug(img_size):
    h = w = img_size
    return A.Compose([
        A.RandomResizedCrop(
            size=(h, w),
            scale=(0.60, 1.0),
            ratio=(0.70, 1.4),
            p=1.0
        ),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.06,
            scale_limit=0.12,
            rotate_limit=12,
            p=0.8
        ),
        A.ColorJitter(0.25, 0.25, 0.25, 0.08, p=0.8),
        A.GaussianBlur(blur_limit=3, p=0.3),
        A.CoarseDropout(
            max_holes=6,
            max_height=int(h * 0.12),
            max_width=int(w * 0.12),
            p=0.4
        ),
        A.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


# ============================================================
# DATASET
# ============================================================
class ImgDataset(Dataset):
    def __init__(self, samples, cls2idx, img_size, aug=True):
        self.samples = samples
        self.cls2idx = cls2idx
        self.img_size = img_size
        self.aug = aug

        self.base_aug = build_aug(img_size, is_train=aug)
        self.strong_aug = build_strong_aug(img_size)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, c = self.samples[i]

        try:
            img = np.array(Image.open(p).convert("RGB"))
        except:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        if self.aug and c in MINORITY_CLASSES:
            data = self.strong_aug(image=img)
        elif self.aug and c in MEDIUM_CLASSES:
            if random.random() < 0.5:
                data = self.strong_aug(image=img)
            else:
                data = self.base_aug(image=img)
        else:
            data = self.base_aug(image=img)

        return data["image"], self.cls2idx[c], p


# ============================================================
# BALANCED SAMPLER
# ============================================================
class BalancedBatchSampler(Sampler):
    def __init__(self, labels, n_cls, n_per):
        self.labels = labels
        self.n_cls = n_cls
        self.n_per = n_per

        self.lab2idx = defaultdict(list)
        for i, l in enumerate(labels):
            self.lab2idx[l].append(i)

        for l in self.lab2idx:
            random.shuffle(self.lab2idx[l])

        self.classes = list(self.lab2idx.keys())
        self.used = {c: 0 for c in self.classes}
        self.num_batches = sum(len(v) for v in self.lab2idx.values()) // (n_cls * n_per)

    def __iter__(self):
        for _ in range(self.num_batches):
            chosen = random.sample(self.classes, self.n_cls)
            batch = []
            for c in chosen:
                st = self.used[c]
                en = st + self.n_per
                if en > len(self.lab2idx[c]):
                    random.shuffle(self.lab2idx[c])
                    st = 0; en = self.n_per
                batch.extend(self.lab2idx[c][st:en])
                self.used[c] = en
            yield batch

    def __len__(self):
        return self.num_batches


# ============================================================
# PROBE MODEL (CLIP → lower classes)
# ============================================================
class ProbeNet(nn.Module):
    def __init__(self, in_dim, hid, ncls):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.fc1 = nn.Linear(in_dim, hid)
        self.fc2 = nn.Linear(hid, hid)
        self.drop = nn.Dropout(0.4)
        self.out = nn.Linear(hid, ncls)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.norm(x)
        x = self.act(self.fc1(x))
        x = self.drop(x)
        x = self.act(self.fc2(x))
        x = self.drop(x)
        return self.out(x)


# ============================================================
# LOAD DATA
# ============================================================
print("Scanning:", ROOT)

samples = collect_samples(ROOT, set(LOWER_CLASSES))

class_counts = Counter(c for _, c in samples)
print("\n=== TOTAL IMAGE COUNT PER CLASS ===")
for cls, cnt in class_counts.most_common():
    print(f"  {cls:20s} : {cnt}")
print(f"  {'TOTAL':20s} : {len(samples)}")

# Train/Val split — 16% val, stratified
by_class = defaultdict(list)
for p, c in samples:
    by_class[c].append((p, c))

train_samples = []
val_samples = []
rng = random.Random(SEED)

for c, arr in by_class.items():
    arr = list(arr)
    rng.shuffle(arr)
    nval = max(2, int(len(arr) * VAL_RATIO))
    val_samples += arr[:nval]
    train_samples += arr[nval:]

print(f"\nTrain: {len(train_samples)}  Val: {len(val_samples)}")

# Save validation split for threshold calibration later
val_split_info = [{"path": p, "label": c} for p, c in val_samples]
val_split_path = OUTDIR / "val_split.json"
with open(val_split_path, "w") as f:
    json.dump(val_split_info, f, indent=2)
print(f"Val split saved to: {val_split_path}  ({len(val_samples)} samples)")


# ============================================================
# PREPARE CLIP EMBEDDINGS (cached to disk)
# ============================================================
def prepare_clip_embeddings(all_samples, cache_path):
    """Pre-compute CLIP embeddings for all samples, cache to disk."""
    if cache_path.exists():
        print(f"Loading cached CLIP embeddings from {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return dict(data)

    print("Preparing CLIP embeddings...")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
    clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=True)

    path2emb = {}
    batch_paths = []
    batch_imgs = []
    batch_size = 32

    with torch.no_grad():
        for i, (p, _) in enumerate(tqdm(all_samples, desc="Computing CLIP embeddings")):
            try:
                img = Image.open(p).convert("RGB")
            except:
                continue

            batch_paths.append(p)
            batch_imgs.append(img)

            if len(batch_imgs) >= batch_size or i == len(all_samples) - 1:
                inputs = clip_proc(images=batch_imgs, return_tensors="pt").to(DEVICE)
                embs = clip_model.get_image_features(**inputs).pooler_output # FIX: Access pooler_output
                embs = embs / embs.norm(dim=-1, keepdim=True)
                embs = embs.cpu().numpy()

                for bp, emb in zip(batch_paths, embs):
                    path2emb[bp] = emb

                batch_paths = []
                batch_imgs = []

    # Save cache
    np.savez(cache_path, **path2emb)
    print(f"CLIP cache saved. ({len(path2emb)} embeddings)")

    del clip_model, clip_proc
    torch.cuda.empty_cache()

    return path2emb


clip_cache_path = OUTDIR / "clip_embeddings.npz"
all_samples_combined = train_samples + val_samples
clip_embeddings = prepare_clip_embeddings(all_samples_combined, clip_cache_path)


# ============================================================
# TRAIN CNN
# ============================================================
def train_lower():
    cls2idx = {c: i for i, c in enumerate(sorted(LOWER_CLASSES))}
    print(f"\nClasses: {cls2idx}")

    tr = train_samples
    vl = val_samples

    cnt = Counter([c for _, c in tr])
    print("Train counts:", dict(cnt))

    labels = [c for _, c in tr]

    # --- Balanced sampler ---
    if all(cnt[c] >= MIN_SAMPLES_FOR_BALANCED for c in cnt):
        ncls = min(len(LOWER_CLASSES), 4)
        nper = max(2, BATCH // ncls)
        sampler = BalancedBatchSampler(labels, ncls, nper)
        tr_ds = ImgDataset(tr, cls2idx, IMG_SIZE, aug=True)
        tr_loader = DataLoader(
            tr_ds,
            batch_sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=True
        )
    else:
        tr_ds = ImgDataset(tr, cls2idx, IMG_SIZE, aug=True)
        weights = [1 / (cnt[c] + 1e-6) for c in labels]
        sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))
        tr_loader = DataLoader(
            tr_ds,
            batch_size=BATCH,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=True
        )

    vl_ds = ImgDataset(vl, cls2idx, IMG_SIZE, aug=False)
    vl_loader = DataLoader(vl_ds, batch_size=24, shuffle=False, num_workers=2)

    # ============================================================
    # CLASS WEIGHTS
    # ============================================================
    class_w = torch.ones(len(cls2idx), device=DEVICE)

    for cls, idx in cls2idx.items():
        n = cnt.get(cls, 0)
        if n > 0:
            class_w[idx] = (1 - 0.9999) / (1 - (0.9999 ** n))
        else:
            class_w[idx] = 1.0

    class_w = class_w / class_w.mean()

    # Boost skirts (minority + confusable)
    boost = {
        "skirts": 1.8,
        "jogging_bottoms": 1.2,
    }

    for cls, factor in boost.items():
        if cls in cls2idx:
            class_w[cls2idx[cls]] *= factor

    class_w = class_w / class_w.mean()

    print(f"\nClass weights:")
    for cls in sorted(cls2idx.keys()):
        print(f"  {cls:20s}: {class_w[cls2idx[cls]]:.4f}")

    # ============================================================
    # CREATE MODEL — ConvNeXt V2 Base
    # ============================================================
    print(f"\nBackbone: {BACKBONE}")
    model = timm.create_model(BACKBONE, pretrained=True, num_classes=len(cls2idx)).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params / 1e6:.1f}M")

    scaler = torch.amp.GradScaler("cuda")

    # ============================================================
    # STAGE 1: TRAIN CLASSIFIER HEAD ONLY
    # ============================================================
    print(f"\n--- Stage 1: Head-only training ({HEAD_EPOCHS} epochs) ---")

    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True

    opt = torch.optim.AdamW(
        model.get_classifier().parameters(),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY
    )

    for ep in range(1, HEAD_EPOCHS + 1):
        model.train()
        losses = []

        for xb, yb, _ in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                out = model(xb)
                loss = focal_loss_with_ls(out, yb, gamma=FOCAL_GAMMA, smooth=LABEL_SMOOTH, weight=class_w)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            losses.append(loss.item())

        print(f"  [head] ep{ep}/{HEAD_EPOCHS}  loss={np.mean(losses):.4f}")

    # ============================================================
    # STAGE 2: FULL FINE-TUNING (ReduceLROnPlateau + MixUp)
    # ============================================================
    print(f"\n--- Stage 2: Full fine-tuning ({EPOCHS} epochs, MixUp α={MIXUP_ALPHA}) ---")

    for p in model.parameters():
        p.requires_grad = True

    head_params = list(model.get_classifier().parameters())
    head_ids = set(map(id, head_params))
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

    opt = torch.optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",          # watching val F1
        patience=6,          # wait 6 epochs before reducing
        factor=0.5,          # halve LR on plateau
        min_lr=1e-7,
    )

    # Store initial LRs for warmup
    initial_lrs = [g["lr"] for g in opt.param_groups]

    best_f1 = -1
    patience_counter = 0

    # ============================================================
    # CRASH-SAFE RESUMABLE CHECKPOINT (helpers + resume)
    # ============================================================
    import shutil as _shutil
    LAST_CKPT_NAME = "best_lower_cnn_last.pth"

    def _atomic_save(obj, final_path):
        tmp = str(final_path) + ".tmp"
        torch.save(obj, tmp)
        os.replace(tmp, final_path)

    def _save_last(state, name, out_dir):
        local = os.path.join("/content", name)
        _atomic_save(state, local)
        drive = os.path.join(str(out_dir), name)
        if drive.endswith(".pt"):     prev = drive[:-3] + "_prev.pt"
        elif drive.endswith(".pth"):  prev = drive[:-4] + "_prev.pth"
        else:                         prev = drive + "_prev"
        try:
            if os.path.exists(drive):
                try: _shutil.copy2(drive, prev)
                except Exception: pass
            tmp = drive + ".tmp"
            _shutil.copy2(local, tmp)
            os.replace(tmp, drive)
        except Exception as e:
            print(f"  [last-ckpt] Drive copy failed (local copy still safe): {e}")

    def _try_load_last(name, out_dir):
        if name.endswith(".pt"):     prev = name[:-3] + "_prev.pt"
        elif name.endswith(".pth"):  prev = name[:-4] + "_prev.pth"
        else:                        prev = name + "_prev"
        for p in [os.path.join("/content", name),
                  os.path.join(str(out_dir), name),
                  os.path.join(str(out_dir), prev)]:
            if not os.path.exists(p): continue
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                _ = ck["epoch"]; _ = ck["model_state"]; _ = ck["optimizer"]
                print(f"  [resume] found checkpoint at {p} (epoch {ck['epoch']})")
                return ck
            except Exception as e:
                print(f"  [resume] corrupt ckpt at {p}: {e}")
        print("  [resume] no resume checkpoint — starting fresh.")
        return None

    # --- Try to resume previous training run ---
    start_ep = 1
    _resume = _try_load_last(LAST_CKPT_NAME, OUTDIR)
    if _resume is not None:
        model.load_state_dict(_resume["model_state"])
        opt.load_state_dict(_resume["optimizer"])
        try: sched.load_state_dict(_resume["scheduler"])
        except Exception as _e: print(f"  [resume] scheduler restore skipped: {_e}")
        try: scaler.load_state_dict(_resume["scaler"])
        except Exception as _e: print(f"  [resume] scaler restore skipped: {_e}")
        start_ep         = _resume["epoch"] + 1
        best_f1          = _resume.get("best_f1", best_f1)
        patience_counter = _resume.get("patience_counter", 0)
        if "rng_torch"  in _resume:
            try: torch.set_rng_state(_resume["rng_torch"])
            except Exception: pass
        if "rng_numpy"  in _resume:
            try: np.random.set_state(_resume["rng_numpy"])
            except Exception: pass
        if "rng_python" in _resume:
            try: random.setstate(_resume["rng_python"])
            except Exception: pass
        if "rng_cuda"   in _resume and torch.cuda.is_available() and _resume["rng_cuda"] is not None:
            try: torch.cuda.set_rng_state_all(_resume["rng_cuda"])
            except Exception: pass
        print(f"  [resume] best_f1={best_f1:.4f}, patience={patience_counter}, next_ep={start_ep}")
        del _resume

    for ep in range(start_ep, EPOCHS + 1):
        model.train()
        losses = []
        n_mixup = 0

        for xb, yb, _ in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                if random.random() < MIXUP_PROB:
                    xb_mixed, ya, yb_mix, lam = mixup_data(xb, yb, alpha=MIXUP_ALPHA)
                    out = model(xb_mixed)
                    loss = focal_loss_mixup(
                        out, ya, yb_mix, lam,
                        gamma=FOCAL_GAMMA, smooth=LABEL_SMOOTH, weight=class_w
                    )
                    n_mixup += 1
                else:
                    out = model(xb)
                    loss = focal_loss_with_ls(
                        out, yb, gamma=FOCAL_GAMMA, smooth=LABEL_SMOOTH, weight=class_w
                    )

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            losses.append(loss.item())

        # LR schedule: warmup then plateau-based (after validation below)
        if ep <= WARMUP_EPOCHS:
            scale = ep / float(WARMUP_EPOCHS)
            for g, init_lr in zip(opt.param_groups, initial_lrs):
                g["lr"] = init_lr * scale

        # ============================================================
        # VALIDATION
        # ============================================================
        model.eval()
        y_true = []
        y_pred = []
        all_logits = []

        with torch.no_grad():
            for xb, yb, _ in vl_loader:
                xb = xb.to(DEVICE, non_blocking=True)

                with torch.amp.autocast("cuda"):
                    logits = model(xb)

                preds = logits.argmax(1).cpu().numpy()
                y_pred += preds.tolist()
                y_true += yb.numpy().tolist()
                all_logits.append(logits.cpu())

        f1 = f1_score(y_true, y_pred, average="macro")
        acc = accuracy_score(y_true, y_pred)
        cur_lr = opt.param_groups[0]["lr"]

        # Step plateau scheduler after warmup
        if ep > WARMUP_EPOCHS:
            old_lr = opt.param_groups[0]["lr"]
            sched.step(f1)
            new_lr = opt.param_groups[0]["lr"]
            if new_lr < old_lr:
                print(f"  ↓ LR reduced: {old_lr:.2e} → {new_lr:.2e}")
                cur_lr = new_lr

        print(f"  [full] ep{ep}/{EPOCHS}  loss={np.mean(losses):.4f}  "
              f"f1={f1:.4f}  acc={acc:.4f}  lr={cur_lr:.2e}  "
              f"mixup={n_mixup}")

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0

            torch.save({
                "model_state": model.state_dict(),
                "classes": sorted(cls2idx.keys()),
                "branch": "lower",
                "backbone": BACKBONE,
                "img_size": IMG_SIZE,
                "best_f1": best_f1,
            }, OUTDIR / "best_lower_cnn.pth")

            # Save validation logits for threshold calibration
            all_logits_cat = torch.cat(all_logits, dim=0)
            all_probs = torch.softmax(all_logits_cat, dim=1).numpy()
            np.savez(
                OUTDIR / "val_predictions.npz",
                y_true=np.array(y_true),
                y_pred=np.array(y_pred),
                probs=all_probs,
                classes=sorted(cls2idx.keys())
            )

            print(f"  ✓ Saved BEST CNN (f1={best_f1:.4f}) + val predictions")
        else:
            patience_counter += 1

        # --- Save resumable "last" checkpoint every epoch ---
        _last_state = {
            "epoch": ep,
            "model_state": model.state_dict(),
            "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict(),
            "scaler": scaler.state_dict(),
            "best_f1": best_f1,
            "patience_counter": patience_counter,
            "rng_torch": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "rng_numpy": np.random.get_state(),
            "rng_python": random.getstate(),
        }
        _save_last(_last_state, LAST_CKPT_NAME, OUTDIR)
        del _last_state

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at ep{ep} (patience={PATIENCE})")
            break

    # ============================================================
    # FINAL CNN REPORT
    # ============================================================
    print(f"\n{'='*60}")
    print(f"  CNN TRAINING COMPLETE — Best F1: {best_f1:.4f}")
    print(f"{'='*60}")

    rep = classification_report(y_true, y_pred, target_names=sorted(cls2idx.keys()), digits=4)
    print(rep)

    (OUTDIR / "lower_cnn_report.txt").write_text(rep)
    np.save(OUTDIR / "lower_cnn_cm.npy", confusion_matrix(y_true, y_pred))

    return best_f1, cls2idx


# ============================================================
# TRAIN PROBE (CLIP → lower classes)
# ============================================================
def train_probe(cls2idx):
    print(f"\n{'='*60}")
    print(f"  TRAINING PROBE (CLIP → lower classes)")
    print(f"{'='*60}")

    ncls = len(cls2idx)

    # Build probe training data from cached CLIP embeddings
    tr_embs, tr_labels = [], []
    for p, c in train_samples:
        if p in clip_embeddings:
            tr_embs.append(clip_embeddings[p])
            tr_labels.append(cls2idx[c])

    vl_embs, vl_labels = [], []
    for p, c in val_samples:
        if p in clip_embeddings:
            vl_embs.append(clip_embeddings[p])
            vl_labels.append(cls2idx[c])

    tr_embs = torch.tensor(np.array(tr_embs), dtype=torch.float32)
    tr_labels = torch.tensor(tr_labels, dtype=torch.long)
    vl_embs = torch.tensor(np.array(vl_embs), dtype=torch.float32)
    vl_labels = torch.tensor(vl_labels, dtype=torch.long)

    print(f"  Probe train: {len(tr_embs)}  Probe val: {len(vl_embs)}")

    # Class weights for probe (same approach)
    cnt = Counter(tr_labels.numpy().tolist())
    probe_w = torch.ones(ncls, device=DEVICE)
    for idx in range(ncls):
        n = cnt.get(idx, 0)
        if n > 0:
            probe_w[idx] = (1 - 0.9999) / (1 - (0.9999 ** n))
        else:
            probe_w[idx] = 1.0

    probe_w = probe_w / probe_w.mean()

    # Skirts boost for probe too
    skirts_idx = cls2idx.get("skirts", -1)
    if skirts_idx >= 0:
        probe_w[skirts_idx] *= 1.5
        probe_w = probe_w / probe_w.mean()

    probe = ProbeNet(512, 768, ncls).to(DEVICE)
    opt = torch.optim.AdamW(probe.parameters(), lr=3e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

    best_probe_f1 = -1
    probe_batch = 128

    for ep in range(100):
        probe.train()
        perm = torch.randperm(len(tr_embs))
        losses = []

        for i in range(0, len(tr_embs), probe_batch):
            idx = perm[i:i+probe_batch]
            xb = tr_embs[idx].to(DEVICE)
            yb = tr_labels[idx].to(DEVICE)

            opt.zero_grad()
            logits = probe(xb)
            loss = focal_loss_with_ls(logits, yb, gamma=FOCAL_GAMMA, smooth=0.05, weight=probe_w)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        sched.step()

        # Validate
        probe.eval()
        with torch.no_grad():
            vl_logits = probe(vl_embs.to(DEVICE))
            vl_preds = vl_logits.argmax(1).cpu().numpy()
            vl_true = vl_labels.numpy()

        pf1 = f1_score(vl_true, vl_preds, average="macro")

        if pf1 > best_probe_f1:
            best_probe_f1 = pf1
            torch.save({
                "probe": probe.state_dict(),
                "classes": sorted(cls2idx.keys()),
            }, OUTDIR / "best_lower_probe.pth")

            if ep % 5 == 0 or ep < 5:
                print(f"  [probe] ep{ep}  loss={np.mean(losses):.4f}  f1={pf1:.4f}")
                print(f"  ✓ Saved BEST PROBE (f1={pf1:.4f})")
        else:
            if ep % 5 == 0:
                print(f"  [probe] ep{ep}  loss={np.mean(losses):.4f}  f1={pf1:.4f}")

    print(f"\n  Probe Best F1: {best_probe_f1:.4f}")

    # Final probe report
    probe.load_state_dict(torch.load(OUTDIR / "best_lower_probe.pth", map_location=DEVICE)["probe"])
    probe.eval()
    with torch.no_grad():
        vl_logits = probe(vl_embs.to(DEVICE))
        vl_preds = vl_logits.argmax(1).cpu().numpy()
        vl_true = vl_labels.numpy()

    rep = classification_report(vl_true, vl_preds, target_names=sorted(cls2idx.keys()), digits=4)
    print(rep)
    (OUTDIR / "lower_probe_report.txt").write_text(rep)

    return best_probe_f1


# ============================================================
# COMPUTE THRESHOLDS
# ============================================================
def compute_thresholds():
    print("\n--- Computing per-class thresholds from validation set ---")

    val_data = np.load(OUTDIR / "val_predictions.npz", allow_pickle=True)
    vt = val_data["y_true"]
    vp = val_data["y_pred"]
    vprobs = val_data["probs"]
    classes = list(val_data["classes"])

    thresholds = {}
    stats = {}
    TARGET_PRECISION = 0.90

    for cls_idx, cls_name in enumerate(classes):
        mask = (vp == cls_idx)
        n_total = mask.sum()

        if n_total < 5:
            thresholds[cls_name] = 0.95
            stats[cls_name] = {"threshold": 0.95, "note": f"Too few predictions ({n_total})"}
            continue

        pred_conf = vprobs[mask, cls_idx]
        pred_correct = (vt[mask] == cls_idx)

        best_t = 0.95
        for t in np.arange(0.20, 0.99, 0.01):
            above = pred_conf >= t
            if above.sum() == 0:
                continue
            precision = pred_correct[above].sum() / above.sum()
            if precision >= TARGET_PRECISION:
                best_t = round(float(t), 2)
                break

        thresholds[cls_name] = best_t
        n_accepted = int((pred_conf >= best_t).sum())
        stats[cls_name] = {
            "threshold": best_t,
            "n_accepted": n_accepted,
            "n_rejected": int(n_total) - n_accepted,
            "n_total": int(n_total),
            "rejection_rate": round((int(n_total) - n_accepted) / max(int(n_total), 1), 4)
        }

    print(f"\n  {'Class':<20s} {'Threshold':>10s} {'Accept':>8s} {'Reject':>8s} {'Rej%':>8s}")
    print(f"  {'-'*55}")
    for cls_name in classes:
        s = stats[cls_name]
        rej = f"{s.get('rejection_rate', 0)*100:.1f}%" if 'rejection_rate' in s else "N/A"
        acc = s.get('n_accepted', 'N/A')
        rej_n = s.get('n_rejected', 'N/A')
        print(f"  {cls_name:<20s} {s['threshold']:>10.2f} {acc:>8} {rej_n:>8} {rej:>8s}")

    thresh_path = OUTDIR / "lower_per_class_thresholds.json"
    with open(thresh_path, "w") as f:
        json.dump({"thresholds": thresholds, "calibration_stats": stats}, f, indent=2)
    print(f"\n  Thresholds saved to: {thresh_path}")


# ============================================================
# RUN
# ============================================================
cnn_f1, cls2idx = train_lower()
probe_f1 = train_probe(cls2idx)
compute_thresholds()

summary = {
    "backbone": BACKBONE,
    "img_size": IMG_SIZE,
    "cnn_best_f1": cnn_f1,
    "probe_best_f1": probe_f1,
    "mixup_alpha": MIXUP_ALPHA,
    "val_ratio": VAL_RATIO,
    "batch_size": BATCH,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
}
(OUTDIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"\nDONE. All outputs saved in: {OUTDIR}")